# GParis 500-user held-out reference

This notebook measures the empirical reference distance used in the Greater Paris row of the comparison table. It compares the 504 users in `gparis_visitation_df.parquet` (the input used by the 500-agent scenario) with a disjoint 500-user cohort from `gparis_visitation_full_data.parquet`.

The holdout ranks eligible users by distinct observed calendar days. Ties are resolved with a seeded random permutation (`seed=42`), so the cohort and all reported metrics are reproducible.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import polars as pl
from IPython.display import display

NOTEBOOK_DIR_CANDIDATES = [
    Path.cwd(),
    Path.cwd() / 'notebooks' / '00_ile_de_france_exploration',
]
NOTEBOOK_DIR = next(
    path for path in NOTEBOOK_DIR_CANDIDATES
    if (path / '05_motif_distribution.ipynb').exists()
)
PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INPUT_PATH = PROJECT_ROOT / 'data/gparis/gparis_visitation_df.parquet'
FULL_PATH = PROJECT_ROOT / 'data/gparis/gparis_visitation_full_data.parquet'
HOLDOUT_PATH = PROJECT_ROOT / 'data/gparis/gparis_visitation_full_data_holdout_500.parquet'
METRICS_PATH = PROJECT_ROOT / 'data/gparis/results/gparis_500_user_holdout_reference_metrics.json'
HOLDOUT_SIZE = 500
TIE_BREAK_SEED = 42

assert INPUT_PATH.exists(), INPUT_PATH
assert FULL_PATH.exists(), FULL_PATH


In [2]:
input_visits = pl.read_parquet(INPUT_PATH)
full_visits = pl.read_parquet(FULL_PATH)

input_ids = input_visits.select('user_id').unique()
full_ids = full_visits.select('user_id').unique()
input_id_set = set(input_ids['user_id'].to_list())
full_id_set = set(full_ids['user_id'].to_list())
assert input_id_set <= full_id_set, 'The 504-user input must be contained in Full GParis.'

eligible = full_visits.join(input_ids, on='user_id', how='anti')
coverage = (
    eligible.with_columns(pl.col('start_timestamp').dt.date().alias('_date'))
    .group_by('user_id')
    .agg(
        pl.col('_date').n_unique().alias('observed_days'),
        pl.len().alias('visit_records'),
    )
    .sort('user_id')
)

# The initial user-ID sort makes the seeded random tiebreak stable across runs.
rng = np.random.default_rng(TIE_BREAK_SEED)
coverage = coverage.with_columns(pl.Series('_tie_break', rng.permutation(coverage.height)))
selected_users = (
    coverage.sort(['observed_days', '_tie_break'], descending=[True, False])
    .head(HOLDOUT_SIZE)
    .sort('user_id')
)
holdout_visits = full_visits.join(selected_users.select('user_id'), on='user_id', how='inner')
holdout_id_set = set(selected_users['user_id'].to_list())

assert len(input_id_set) == 504
assert len(holdout_id_set) == HOLDOUT_SIZE
assert input_id_set.isdisjoint(holdout_id_set)
assert selected_users['observed_days'].min() == 7
assert selected_users.filter(pl.col('observed_days') == 8).height == 4
assert selected_users.filter(pl.col('observed_days') == 7).height == 496

HOLDOUT_PATH.parent.mkdir(parents=True, exist_ok=True)
holdout_visits.write_parquet(HOLDOUT_PATH)

cohort_summary = pd.DataFrame([
    {'cohort': 'GParis input', 'users': len(input_id_set), 'visit_records': input_visits.height},
    {
        'cohort': 'Full GParis holdout',
        'users': len(holdout_id_set),
        'visit_records': holdout_visits.height,
        'min_observed_days': selected_users['observed_days'].min(),
        'max_observed_days': selected_users['observed_days'].max(),
        'mean_observed_days': selected_users['observed_days'].mean(),
    },
])
display(cohort_summary)
display(selected_users.group_by('observed_days').len().sort('observed_days', descending=True))


,cohort,users,visit_records,min_observed_days,max_observed_days,mean_observed_days
0,GParis input,504,13797,NaN,NaN,NaN
1,Full GParis holdout,500,12898,7.0,8.0,7.008


observed_days,len
u32,u32
8,4
7,496


In [3]:
from citybehavex.reports.comparison import generate_comparison_report_from_paths

# Core Wasserstein metrics always run. activity_jsd supplies VPD, ATM, and DARD;
# road-network distance remains off to match configs/gparis_simulation.yaml.
generate_comparison_report_from_paths(
    synthetic_path=str(INPUT_PATH),
    real_path=str(HOLDOUT_PATH),
    observed_label='GParis Full holdout (500 users)',
    json_output_path=str(METRICS_PATH),
    sections=['activity_jsd'],
)

metrics = json.loads(METRICS_PATH.read_text())
table_metrics = pd.DataFrame([
    {'metric': r'$\Delta r$ ($W_1$, km)', 'value': metrics['wasserstein']['jump_lengths_km']},
    {'metric': r'$r_g$ ($W_1$, km)', 'value': metrics['wasserstein']['radius_of_gyration_km']},
    {'metric': 'TD ($W_1$, min)', 'value': metrics['wasserstein']['trip_duration_min']},
    {'metric': 'DT ($W_1$, h)', 'value': metrics['wasserstein']['dwell_time_min'] / 60.0},
    {'metric': r'$V_f$ ($W_1$)', 'value': metrics['wasserstein']['visits_per_user']},
    {'metric': r'VPD ($10^{-2}$)', 'value': metrics['jsd']['activity_distribution'] * 100.0},
    {'metric': r'ATM ($10^{-2}$)', 'value': metrics['jsd']['activity_transitions'] * 100.0},
    {'metric': r'DARD ($10^{-2}$)', 'value': metrics['jsd']['daily_activity_profile'] * 100.0},
])
assert np.isfinite(table_metrics['value']).all()
display(table_metrics.round(4))


Loading synthetic trajectories from /home/gustavo/citybehavex/data/gparis/gparis_visitation_df.parquet ...


Loading observed trajectories from /home/gustavo/citybehavex/data/gparis/gparis_visitation_full_data_holdout_500.parquet ...


Computing mobility metrics ...


Comparison metrics -> /home/gustavo/citybehavex/data/gparis/results/gparis_500_user_holdout_reference_metrics.json  (Jump lengths: 1.0407  Visits per user: 1.0120  Radius of gyration: 1.4570  Dwell time: 35.7709  Trip duration: 1.2489)


,metric,value
0,"$\Delta r$ ($W_1$, km)",1.0407
1,"$r_g$ ($W_1$, km)",1.4570
2,"TD ($W_1$, min)",1.2489
3,"DT ($W_1$, h)",0.5962
4,$V_f$ ($W_1$),1.0120
5,VPD ($10^{-2}$),1.8034
6,ATM ($10^{-2}$),3.6249
7,DARD ($10^{-2}$),1.8858


The displayed values are copied as single point estimates into the `GParis / Ref.` row of `paper/comparision_table.tex`. They do not carry a standard deviation because this is one fixed, disjoint reference comparison.